use the Evidence Lower Bound (ELBO) to determine the number of hidden states in a Hidden Markov Model (HMM) for each time series, and to take advantage of the fact that the time series may be correlated, potentially coming from a similar Gaussian mixture, to simplify implementation and computation.

One way to do this is to use a hierarchical Bayesian model, in which the parameters of the HMM for each time series are drawn from a shared distribution. Specifically, we can assume that the means and covariances of the Gaussian emissions for each time series are drawn from a common multivariate normal distribution with hyperparameters, and that the transition matrix and initial state distribution for each time series are drawn from a common Dirichlet distribution with hyperparameters.

We can then use Bayesian inference to estimate the hyperparameters of the shared distributions, and to infer the optimal number of hidden states for each time series, given the shared distributions. We can use the ELBO to perform variational inference and approximate the posterior distributions of the hidden states and hyperparameters.

Here's an example of how to implement this procedure in Python using the Pyro library:

In [ ]:
!pip install pyro --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 6.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import numpy as np
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam

# Define the number of time series and the number of features
n_series = 10
n_features = 3

# Generate some example data
n_samples = 100
data = np.random.randn(n_series, n_samples, n_features)

# Define the priors for the hyperparameters
mean_prior = dist.Normal(torch.zeros(n_features), 10*torch.ones(n_features))
cov_prior = dist.InverseWishart(n_features+1, torch.eye(n_features))
alpha_prior = torch.ones(2)

# Define the model for each time series
def hmm_model(data):
    n_states = pyro.sample('n_states', dist.Categorical(alpha_prior))
    init_state_prior = dist.Dirichlet(0.5*torch.ones(n_states))
    trans_mat_prior = dist.Dirichlet(0.5*torch.ones(n_states, n_states))
    pi = pyro.sample('pi', init_state_prior)
    A = pyro.sample('A', trans_mat_prior)
    means = pyro.sample('means', mean_prior.expand((n_states, n_features)))
    covs = pyro.sample('covs', cov_prior.expand(n_states))
    with pyro.plate('data', len(data)):
        x = None
        z = None
        for t in range(len(data)):
            if x is None:
                x = pyro.sample('x_{}'.format(t), dist.Categorical(pi))
            else:
                x = pyro.sample('x_{}'.format(t), dist.Categorical(A[x]))
            if z is None:
                z = pyro.sample('z_{}'.format(t), dist.MultivariateNormal(means[x], covs[x]).expand([len(data),]))
            else:
                z = pyro.sample('z_{}'.format(t), dist.MultivariateNormal(means[x], covs[x]).mask(data[t].bool()))

# Define the guide for each time series
def hmm_guide(data):
    n_states = pyro.param('n_states', torch.tensor([2.0]*n_series), constraint=dist.constraints.simplex)
    pi = pyro.param('pi', torch.ones(n_series, 2)/2, constraint=dist.constraints.simplex)
    A = pyro.param('A', torch.ones(n_series, 2, 2)/2, constraint=dist.constraints.simplex)
    means_loc = pyro.param('means


SyntaxError: ignored

Here, we generate some example data consisting of 10 time series, each with 1000 time steps and 5 features. We then initialize an HDPHMM model with a maximum of 10 states, and fit the model to the data using the fit method. Finally, we extract the inferred number of states for each time series using the predict method, and print out the results.

Note that the HDPHMM model allows for shared states across time series, but still allows for differences in the emission distributions for each time series. If the time series share a common Gaussian mixture, you could use a Bayesian nonparametric model that explicitly models the shared mixture, such as the Hierarchical Dirichlet Process Gaussian mixture model (HDP-GMM). However, implementing such a model may be more complex and computationally demanding than the HDP-HMM.

In [ ]:
import numpy as np
from hmmlearn import hmm

# Generate some example data
n_samples = 1000
n_features = 5
n_series = 10
data = [np.random.randn(n_samples, n_features) for _ in range(n_series)]

# Initialize the HDP-HMM
model = hmm.GaussianHDPHMM(n_components=10)

# Fit the model to the data
model.fit(data)

# Extract the learned number of states for each time series
n_states = model.predict(data)

# Print the inferred number of states for each time series
print(f"Inferred number of states: {n_states}")


Yes, it is possible to use the Evidence Lower Bound (ELBO) to determine the number of hidden states for each time series, and to take advantage of the correlation between the time series to simplify implementation and computation.

One approach to doing this is to use a shared Gaussian mixture model (GMM) to model the emission distribution for each time series. The GMM can be shared across all time series, allowing for correlations between the time series to be taken into account. The number of components in the shared GMM can be determined using a separate procedure, such as the Bayesian Information Criterion (BIC).

Once the shared GMM has been trained, it can be used to compute the emission probabilities for each time series. The ELBO can then be used to determine the number of hidden states for each time series, using a similar procedure to the one described in my previous response. However, instead of training a separate GMM for each time series, the shared GMM can be used to compute the emission probabilities, simplifying the implementation and computation.

Here's an example of how to implement this procedure in Python, using the hmmlearn library:
-Here, we first generate some example data consisting of n_series time series, each with n_samples observations of n_features variables. We fit a shared Gaussian mixture model to the data using the GaussianMixture class from scikit-learn. We then loop over each time series and the range of numbers of states to consider. For each combination of time series and number of states, we initialize an HMMGaussian model and compute the emission probabilities for the time series using the shared GMM. We fit the model on the emission probabilities, compute the log-likelihood and the ELBO, and print the ELBO for each model.

Note that in practice, you may need to adjust the parameters of the model, the shared GMM, and the range of numbers of states to consider to get good results for your specific application. Additionally, you may want to experiment with different procedures for determining the number of components in the shared GMM, such as the B

In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture
from hmmlearn import hmm

# Generate some example data
n_samples = 100
n_features = 3
n_series = 5
data = np.random.randn(n_series, n_samples, n_features)

# Fit a shared Gaussian mixture model to the data
n_components = 5
gmm = GaussianMixture(n_components=n_components)
gmm.fit(data.reshape(-1, n_features))

# Define the range of numbers of states to consider
min_states = 2
max_states = 5

# Loop over the time series
for i in range(n_series):
    print(f"Time series {i+1}:")

    # Loop over the range of numbers of states
    for n_states in range(min_states, max_states+1):
        print(f"  Training model with {n_states} states...")

        # Initialize the model
        model = hmm.GaussianHMM(n_components=n_states, covariance_type='diag')

        # Compute the emission probabilities for this time series
        probs = gmm.predict_proba(data[i])

        # Fit the model on the emission probabilities
        model.fit(probs)

        # Compute the log-likelihood of the data under the model
        log_likelihood = model.score(probs)

        # Compute the ELBO for the model
        elbo = log_likelihood - model.score_samples(probs).mean()

        # Print the ELBO for this model
        print(f"    ELBO = {elbo:.3f}")
